# 12 — SBI foundations: ABC and neural posterior estimation

**Question.** If we can simulate data but cannot evaluate the
likelihood, how can we infer the parameters that generated an
observation?

We use the deliberately simple simulator

$$
\theta \sim \mathcal U(-3,3), \qquad
x = \theta^2 + \epsilon, \qquad
\epsilon \sim \mathcal N(0,0.25^2).
$$

For $x_o=4$, predict the posterior before running any code. Is it
centred near zero, one-sided, or bimodal? Explain your prediction
from the forward map.

**Learning goals**

- implement rejection ABC and explain the tolerance trade-off;
- train a small conditional mixture-density NPE from scratch;
- use one amortized estimator for several observations;
- repeat the inference with the public `sbi` package; and
- distinguish posterior predictive fit from repeated-simulation
  coverage.

In [2]:
from pathlib import Path
import json
import os
import random
import sys
import time
import warnings

ROOT = Path(os.path.abspath('.'))

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(OUTPUT_DIR / ".mplconfig"))

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

SEED = 2605
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)
warnings.filterwarnings("ignore", message="IProgress not found.*")

class NullTracker:
    '''Minimal no-op tracker that keeps classroom runs out of TensorBoard.'''

    @property
    def log_dir(self):
        return None

    def log_metric(self, name, value, step=None):
        pass

    def log_metrics(self, metrics, step=None):
        pass

    def log_params(self, params):
        pass

    def add_figure(self, name, figure, step=None):
        pass

    def flush(self):
        pass

COLORS = {
    "theta": "#7656A5",
    "data": "#2A9D8F",
    "observation": "#C94C4C",
    "learned": "#E6862E",
    "reference": "#2D6A9F",
    "gray": "#626C78",
}

mpl.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 180,
    "figure.facecolor": "white",
    "axes.facecolor": "#FBFCFE",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 10.5,
    "legend.frameon": False,
})

def savefig(fig, name):
    fig.text(
        0.995, 0.005, "analytic teaching model",
        ha="right", va="bottom", fontsize=7, color=COLORS["gray"],
    )
    path = OUTPUT_DIR / name
    fig.savefig(path, bbox_inches="tight", facecolor="white")
    print(f"saved: {path.relative_to(ROOT)}")
    return path

def write_json(name, payload):
    path = OUTPUT_DIR / name
    with path.open("w") as stream:
        json.dump(payload, stream, indent=2, sort_keys=True)
        stream.write("\n")
    print(f"saved: {path.relative_to(ROOT)}")
    return path

In [3]:
import torch
from torch import nn
import torch.nn.functional as F
from torch.distributions import Categorical, Normal
from torch.utils.data import DataLoader, TensorDataset

from teaching_sbi import (
    TOY_NOISE_SIGMA,
    exact_toy_posterior,
    sample_toy_prior,
    simulate_toy_reference,
)

torch.manual_seed(SEED)
torch.set_num_threads(1)
NOISE_SIGMA = TOY_NOISE_SIGMA
X_OBSERVED = 4.0
THETA_GRID = np.linspace(-3.0, 3.0, 1601)
EXACT_AT_XO = exact_toy_posterior(
    THETA_GRID, X_OBSERVED, noise_sigma=NOISE_SIGMA
)

## 1. The simulator defines the likelihood implicitly

A simulator needs to draw $x\sim p(x\mid\theta)$; it need not
return the numerical value of $p(x\mid\theta)$. Implement the
forward model below. We compare it with an independent reference
function only to catch a coding error in this teaching exercise.

In [ ]:
def simulate_toy(theta, rng, noise_sigma=NOISE_SIGMA):
    '''Simulate x = theta^2 + Gaussian noise for a batch of theta values.'''
    theta = np.asarray(theta, dtype=float)
    if theta.ndim == 1:
        theta = theta[:, None]

    # TODO 1:  draw independent Gaussian noise with theta.shape and
    # return x = theta**2 + noise. Keep the output shape (n, 1).
    raise NotImplementedError

In [ ]:
theta_check = np.array([[-2.0], [0.0], [1.5]])
check_a = simulate_toy(theta_check, np.random.default_rng(11))
check_b = simulate_toy_reference(theta_check, np.random.default_rng(11))
np.testing.assert_allclose(check_a, check_b)

n_simulations = 8_000
theta_train = sample_toy_prior(n_simulations, rng)
x_train = simulate_toy(theta_train, rng)

fig, ax = plt.subplots(figsize=(8.8, 4.8), constrained_layout=True)
shown = rng.choice(n_simulations, size=2_000, replace=False)
ax.scatter(
    theta_train[shown, 0], x_train[shown, 0],
    s=7, alpha=0.25, color=COLORS["data"], rasterized=True,
)
ax.axhline(
    X_OBSERVED, color=COLORS["observation"], lw=2,
    label=rf"observation $x_o={X_OBSERVED:g}$",
)
ax.set(xlabel=r"$\theta$", ylabel=r"$x$", ylim=(-0.7, 9.8))
ax.set_title("The posterior is a horizontal conditional slice")
ax.legend()
savefig(fig, "12_toy_joint.png")
plt.show()

## 2. Rejection ABC

Draw $(\theta_i,x_i)$ from the joint distribution and retain
$\theta_i$ when

$$
\rho(x_i,x_o)\leq\varepsilon.
$$

These accepted values approximate a posterior conditioned on a
tolerance region. A smaller $\varepsilon$ reduces that
approximation but also reduces the acceptance rate.

In [ ]:
def rejection_abc(theta_simulated, x_simulated, x_observed, epsilon):
    '''Return theta values whose scalar summaries lie within epsilon of x_o.'''
    # TODO 2: compute |x_i - x_o| and return the accepted theta_i.
    # This is a one-dimensional summary, so no extra norm is needed.
    raise NotImplementedError

In [ ]:
epsilons = [1.0, 0.5, 0.2, 0.08]
fig, axes = plt.subplots(
    1, len(epsilons), figsize=(13.0, 3.3),
    sharex=True, sharey=True, constrained_layout=True,
)
abc_summary = {}
for ax, epsilon in zip(axes, epsilons):
    accepted = rejection_abc(
        theta_train, x_train, X_OBSERVED, epsilon
    )
    abc_summary[str(epsilon)] = {
        "accepted": int(accepted.size),
        "acceptance_rate": float(accepted.size / n_simulations),
    }
    ax.hist(
        accepted, bins=45, range=(-3, 3), density=True,
        color=COLORS["theta"], alpha=0.52,
    )
    ax.plot(
        THETA_GRID, EXACT_AT_XO,
        color=COLORS["reference"], lw=1.8, label="exact",
    )
    ax.set_title(
        rf"$\varepsilon={epsilon:g}$"
        + f"\naccept {accepted.size / n_simulations:.1%}"
    )
    ax.set_xlabel(r"$\theta$")
axes[0].set_ylabel("density")
axes[0].legend()
fig.suptitle("ABC: accuracy and simulation efficiency trade off")
savefig(fig, "12_abc_tolerance.png")
plt.show()

## 3. NPE learns a conditional density

Neural posterior estimation fits

$$
q_\phi(\theta\mid x)\approx p(\theta\mid x)
$$

by minimizing

$$
\mathcal L_{\rm NPE}
=-\mathbb E_{p(\theta,x)}
  \left[\log q_\phi(\theta\mid x)\right].
$$

A point-regression network would average the two inverse branches
near $\theta=\pm\sqrt{x_o}$ and could return the low-probability
value zero. Instead, our network outputs the weights, means, and
scales of a Gaussian mixture.

In [ ]:
class ConditionalMDN(nn.Module):
    '''Three-component Gaussian mixture q_phi(theta | x).'''

    def __init__(self, n_components, x_mean, x_std):
        super().__init__()
        self.n_components = n_components
        self.register_buffer("x_mean", torch.as_tensor(x_mean, dtype=torch.float32))
        self.register_buffer("x_std", torch.as_tensor(x_std, dtype=torch.float32))
        self.network = nn.Sequential(
            nn.Linear(1, 48), nn.Tanh(),
            nn.Linear(48, 48), nn.Tanh(),
            nn.Linear(48, 3 * n_components),
        )

    def mixture_parameters(self, x):
        raw = self.network((x - self.x_mean) / self.x_std)
        logits, raw_means, raw_scales = torch.chunk(raw, 3, dim=-1)

        # TODO 3a: map means to roughly the prior support [-3, 3].
        # Make every standard deviation positive with softplus and a small
        # positive floor. Return logits, means, scales.
        raise NotImplementedError

    def log_prob(self, theta, x):
        logits, means, scales = self.mixture_parameters(x)

        # TODO 3b: evaluate every Gaussian component, add normalized
        # log mixture weights, and use torch.logsumexp over components.
        raise NotImplementedError

In [ ]:
def npe_loss(model, theta_batch, x_batch):
    '''Monte-Carlo estimate of -E_joint[log q_phi(theta | x)].'''
    # TODO 4: return the negative mean conditional log probability.
    raise NotImplementedError

In [ ]:
theta_tensor = torch.as_tensor(theta_train, dtype=torch.float32)
x_tensor = torch.as_tensor(x_train, dtype=torch.float32)
loader = DataLoader(
    TensorDataset(theta_tensor, x_tensor),
    batch_size=256, shuffle=True,
    generator=torch.Generator().manual_seed(SEED),
)

model = ConditionalMDN(
    n_components=3,
    x_mean=x_train.mean(axis=0),
    x_std=x_train.std(axis=0),
)
optimizer = torch.optim.Adam(model.parameters(), lr=2.0e-3)
losses = []
started = time.perf_counter()
for epoch in range(140):
    epoch_losses = []
    for theta_batch, x_batch in loader:
        optimizer.zero_grad()
        loss = npe_loss(model, theta_batch, x_batch)
        loss.backward()
        optimizer.step()
        epoch_losses.append(float(loss.detach()))
    losses.append(float(np.mean(epoch_losses)))
print(f"training time: {time.perf_counter() - started:.1f} s")
print(f"final NPE loss: {losses[-1]:.3f}")

fig, ax = plt.subplots(figsize=(7.5, 3.8), constrained_layout=True)
ax.plot(losses, color=COLORS["learned"], lw=2)
ax.set(
    xlabel="epoch",
    ylabel=r"$-\langle\log q_\phi(\theta\mid x)\rangle$",
    title="Conditional-density training",
)
savefig(fig, "12_mdn_training.png")
plt.show()

In [ ]:
@torch.no_grad()
def sample_mdn(model, x_value, n_samples, seed):
    '''Draw scalar theta samples from q_phi(theta | x_value).'''
    generator = torch.Generator().manual_seed(seed)
    x_value = torch.tensor([[x_value]], dtype=torch.float32)
    logits, means, scales = model.mixture_parameters(x_value)
    components = torch.multinomial(
        torch.softmax(logits[0], dim=-1),
        n_samples, replacement=True, generator=generator,
    )
    selected_means = means[0, components]
    selected_scales = scales[0, components]
    noise = torch.randn(n_samples, generator=generator)
    return (selected_means + selected_scales * noise).numpy()

mdn_samples = sample_mdn(model, X_OBSERVED, 8_000, SEED + 1)
inside_prior = (mdn_samples >= -3.0) & (mdn_samples <= 3.0)
leakage = 1.0 - inside_prior.mean()

fig, ax = plt.subplots(figsize=(8.8, 4.2), constrained_layout=True)
ax.hist(
    mdn_samples[inside_prior], bins=70, range=(-3, 3), density=True,
    color=COLORS["learned"], alpha=0.52, label="from-scratch NPE",
)
ax.plot(
    THETA_GRID, EXACT_AT_XO,
    color=COLORS["reference"], lw=2.2, label="exact posterior",
)
ax.set(
    xlabel=r"$\theta$", ylabel="density",
    title=rf"Posterior at $x_o={X_OBSERVED:g}$; "
          + f"raw Gaussian-tail leakage = {leakage:.2%}",
)
ax.legend()
savefig(fig, "12_toy_posterior.png")
plt.show()

The small Gaussian components have non-zero tails outside the
uniform prior. This is an estimator-support issue, not a new
physical possibility. The high-level `sbi` posterior below
enforces the prior support when drawing samples.

## 4. Amortization

The model was trained on the full prior predictive distribution,
not only at $x_o$. We can therefore query new observations without
new simulations or retraining.

In [ ]:
query_x = [0.5, 2.0, 4.0]
fig, axes = plt.subplots(
    1, 3, figsize=(12.0, 3.4), sharex=True, sharey=True,
    constrained_layout=True,
)
for index, (ax, x_value) in enumerate(zip(axes, query_x)):
    samples = sample_mdn(model, x_value, 5_000, SEED + 10 + index)
    samples = samples[(samples >= -3) & (samples <= 3)]
    exact = exact_toy_posterior(THETA_GRID, x_value)
    ax.hist(
        samples, bins=55, range=(-3, 3), density=True,
        color=COLORS["learned"], alpha=0.48,
    )
    ax.plot(THETA_GRID, exact, color=COLORS["reference"], lw=1.8)
    ax.set_title(rf"$x_o={x_value:g}$")
    ax.set_xlabel(r"$\theta$")
axes[0].set_ylabel("density")
fig.suptitle("One trained NPE, three posterior queries")
savefig(fig, "12_amortized_queries.png")
plt.show()

## 5. The same workflow with `sbi`

The package call has the same conceptual steps as our manual
implementation: append joint simulations, train a conditional
density estimator, build a posterior object, and condition it on
the observation. Here we use an MDN because the problem is
one-dimensional and explicitly multimodal.

In [ ]:
from sbi.inference import NPE
from sbi.utils import BoxUniform

sbi_prior = BoxUniform(
    low=torch.tensor([-3.0]),
    high=torch.tensor([3.0]),
)
inference = NPE(
    prior=sbi_prior,
    density_estimator="mdn",
    tracker=NullTracker(),
    show_progress_bars=False,
)
density_estimator = (
    inference
    .append_simulations(theta_tensor, x_tensor)
    .train(
        training_batch_size=256,
        max_num_epochs=80,
        stop_after_epochs=10,
        show_train_summary=False,
    )
)
sbi_posterior = inference.build_posterior(density_estimator)
sbi_samples = (
    sbi_posterior.sample(
        (8_000,),
        x=torch.tensor([X_OBSERVED]),
        show_progress_bars=False,
    )
    .cpu().numpy()[:, 0]
)

fig, ax = plt.subplots(figsize=(8.8, 4.2), constrained_layout=True)
ax.hist(
    sbi_samples, bins=70, range=(-3, 3), density=True,
    color=COLORS["theta"], alpha=0.42, label="sbi NPE",
)
ax.plot(
    THETA_GRID, EXACT_AT_XO,
    color=COLORS["reference"], lw=2.2, label="exact",
)
ax.set(
    xlabel=r"$\theta$", ylabel="density",
    title="High-level and analytic posteriors agree on both modes",
)
ax.legend()
savefig(fig, "12_sbi_package_posterior.png")
plt.show()

## 6. Diagnostics answer different questions

A posterior predictive check draws

$$
\theta^{(m)}\sim q_\phi(\theta\mid x_o), \qquad
x_{\rm rep}^{(m)}\sim p(x\mid\theta^{(m)}).
$$

It checks whether posterior-supported parameters can reproduce the
observed summary. It does **not** by itself establish calibrated
uncertainty.

In [ ]:
x_replicated = simulate_toy(
    sbi_samples[:, None], np.random.default_rng(SEED + 40)
)[:, 0]
fig, ax = plt.subplots(figsize=(8.6, 4.0), constrained_layout=True)
ax.hist(
    x_replicated, bins=65, density=True,
    color=COLORS["data"], alpha=0.55,
    label=r"$x_{\rm rep}$",
)
ax.axvline(
    X_OBSERVED, color=COLORS["observation"], lw=2.4,
    label=r"$x_o$",
)
ax.set(
    xlabel=r"replicated $x$", ylabel="density",
    title="Posterior predictive check",
)
ax.legend()
savefig(fig, "12_posterior_predictive.png")
plt.show()

In [ ]:
def marginal_coverage(posterior_samples, theta_true, probability):
    '''Equal-tailed marginal coverage for scalar theta.'''
    # TODO 5: form the central interval for every simulated case,
    # check whether its generating theta lies inside, and return the fraction.
    raise NotImplementedError

In [ ]:
n_cases = 120
theta_test = sample_toy_prior(
    n_cases, np.random.default_rng(SEED + 50)
)[:, 0]
x_test = simulate_toy(
    theta_test[:, None], np.random.default_rng(SEED + 51)
)[:, 0]
posterior_test = np.stack([
    sample_mdn(model, x_value, 600, SEED + 100 + index)
    for index, x_value in enumerate(x_test)
])

levels = np.array([0.50, 0.68, 0.80, 0.95])
coverages = np.array([
    marginal_coverage(posterior_test, theta_test, level)
    for level in levels
])
standard_error = np.sqrt(levels * (1.0 - levels) / n_cases)

fig, ax = plt.subplots(figsize=(5.7, 5.0), constrained_layout=True)
ax.errorbar(
    levels, coverages, yerr=standard_error,
    fmt="o", color=COLORS["learned"], capsize=4,
    label="from-scratch NPE",
)
ax.plot([0.4, 1.0], [0.4, 1.0], "--", color=COLORS["gray"])
ax.set(
    xlim=(0.45, 0.98), ylim=(0.45, 0.98),
    xlabel="nominal interval probability",
    ylabel="empirical coverage",
    title="Coverage requires repeated simulated observations",
)
ax.legend()
savefig(fig, "12_coverage.png")
plt.show()

write_json(
    "12_sbi_summary.json",
    {
        "seed": SEED,
        "n_training_simulations": n_simulations,
        "abc": abc_summary,
        "mdn_tail_leakage": float(leakage),
        "coverage_levels": levels.tolist(),
        "empirical_coverage": coverages.tolist(),
        "n_coverage_cases": n_cases,
    },
)

## What this tutorial established

- The two posterior modes come from the many-to-one forward model,
  not from the neural estimator.
- ABC exposes the basic logic of likelihood-free inference but
  becomes inefficient as the matching problem grows.
- NPE learns an entire conditional distribution rather than a
  point prediction.
- Amortization moves work into a shared simulation and training
  phase.
- A posterior plot, a posterior predictive check, and coverage are
  different pieces of evidence.

**Short reflection.** If the observation were far outside the
prior-predictive range, why might a neural posterior still look
smooth? What check should happen before interpreting it?

**Sources**

- [A Practical Guide to Simulation-Based Inference](https://arxiv.org/abs/2508.12939)
- [`sbi` getting-started tutorial](https://sbi.readthedocs.io/en/latest/tutorials/00_getting_started.html)
- [`sbi` Bayesian workflow](https://sbi.readthedocs.io/en/latest/tutorials/01_Bayesian_workflow.html)
- [Papamakarios & Murray (2016), Fast epsilon-free inference of
  simulation models with Bayesian conditional density
  estimation](https://arxiv.org/abs/1605.06376)
- [Talts et al. (2018), Validating Bayesian Inference Algorithms
  with Simulation-Based Calibration](https://arxiv.org/abs/1804.06788)